In [ ]:
try:
    import claveles  # noqa: F401
except ImportError:
    %pip install claveles

**Note for Google Colab Users:**
During installation, Colab may display a dependency conflict error regarding `protobuf` (`ERROR: pip's dependency resolver...`). This occurs due to pre-installed Google packages in the Colab environment. 

As long as you see `Successfully installed ...` at the end of the log, the required packages are properly installed. **Please ignore the error and continue running the rest of the notebook.**

$$
\newcommand{\bra}[1]{{\left\langle{#1}\right|}}
\newcommand{\ket}[1]{{\left|{#1}\right\rangle}}
\newcommand{\Bra}[1]{{\Big\langle{#1}\Big|}}
\newcommand{\Ket}[1]{{\Big|{#1}\Big\rangle}}
\newcommand{\braket}[1]{{\left\langle{#1}\right\rangle}}
\newcommand{\dag}{\dagger}
$$

# Generating EPR States with a CZ Gate: Comparison with the Beamsplitter

## Introduction

A two-mode EPR state can be generated in multiple ways.
While the first notebook constructed it using a 50:50 beamsplitter, this tutorial establishes the *same* EPR correlations using a continuous-variable (CV) Controlled-Z (CZ) gate and provides a quantitative comparison between the two techniques.

Although both approaches satisfy the conditions $\hat{x}_0 \approx \hat{x}_1$ and $\hat{p}_0 \approx -\hat{p}_1$, the residual variance of these correlations differs by a factor of two.
We validate this difference against theoretical predictions and sweep the squeezing parameter to observe how the correlation quality scales.

> **Prerequisite:** This notebook builds upon [Two-Mode EPR State: Generation and Quantum Correlations](./epr1.ipynb), which introduces the fundamentals of EPR state generation and quantum correlations. We highly recommend reading that notebook first.
>
> This is one of two independent follow-up notebooks; the other explores heralding non-Gaussian states via Fock-number measurements. Both follow-up notebooks depend exclusively on the foundational first notebook and can be read in any order.

---

## Background

### EPR Difference and Sum Quadratures (recap)

A two-mode EPR state satisfies $\hat{x}_0 - \hat{x}_1 \to 0$ and $\hat{p}_0 + \hat{p}_1 \to 0$ in the limit of infinite squeezing ($r\to\infty$).
Consequently, the quality of a physically realizable, approximate EPR state is quantified by the variances of the **difference and sum quadratures**, $x_0 - x_1$ and $p_0 + p_1$: smaller variances indicate closer proximity to the ideal state.
The beamsplitter method (detailed in the first notebook) achieves this by mixing an $x$-squeezed vacuum and a $p$-squeezed vacuum on a beamsplitter $\hat{B}(\pi/4, \pi)$.

### Generation by CZ Gate

Alternatively, two $p$-squeezed vacuum modes can be coupled using a CV CZ gate:

$$
\widehat{\mathrm{CZ}}(g) = \exp\!\left(i\frac{g}{\hbar}\hat{x}_0\hat{x}_1\right),
$$

which transforms the quadrature operators as follows:

$$
(\hat{x}_0, \hat{p}_0, \hat{x}_1, \hat{p}_1) \xrightarrow{\widehat{\mathrm{CZ}}(1)} (\hat{x}_0,\; \hat{p}_0 + \hat{x}_1,\; \hat{x}_1,\; \hat{p}_1 + \hat{x}_0).
$$

Applying a subsequent phase rotation $\hat{R}(-\pi/2)$ to mode 1 maps the quadratures as $(\hat{x}, \hat{p}) \to (\hat{p}, -\hat{x})$, yielding:

$$
(\hat{x}_0,\; \hat{p}_0 + \hat{x}_1,\; \hat{x}_1,\; \hat{p}_1 + \hat{x}_0)
\xrightarrow{\hat{I}\otimes\hat{R}(-\pi/2)} (\hat{x}_0', \hat{p}_0', \hat{x}_1', \hat{p}_1') = (\hat{x}_0,\; \hat{p}_0 + \hat{x}_1,\; \hat{p}_1 + \hat{x}_0,\; -\hat{x}_1).
$$

When both initial modes are $p$-squeezed with squeezing level $r$ ($\hat{p}_0 = e^{-r}\hat{p}_{0,\mathrm{vac}}$ and $\hat{p}_1 = e^{-r}\hat{p}_{1,\mathrm{vac}}$), the difference and sum quadratures simplify to:

$$
\hat{x}_0' - \hat{x}_1' = -e^{-r}\hat{p}_{1,\mathrm{vac}} \to 0, \qquad \hat{p}_0' + \hat{p}_1' = e^{-r}\hat{p}_{0,\mathrm{vac}} \to 0
$$

in the limit as $r\to\infty$.

### Why the Variances Differ by a Factor of Two

While both techniques yield identical EPR correlations ($\hat{x}_0 \approx \hat{x}_1$ and $\hat{p}_0 \approx -\hat{p}_1$) in the limit of infinite squeezing, the variance of $x_0 - x_1$ (and $p_0 + p_1$) for the CZ method is precisely **half** that of the beamsplitter method.
In the CZ configuration, each correlation relation is carried directly by a single squeezed quadrature, whereas the beamsplitter routes a $\sqrt{2}$-amplified component of the squeezed quadratures into each output mode.
Quantitatively, the resulting variance is $e^{-2r}\hbar$ for the beamsplitter method (see [EPR Tutorial 1](./epr1.ipynb)) and $e^{-2r}\hbar/2$ for the CZ gate method.

However, this fact does not necessarily imply that the CZ gate approach is practically more advantageous. In fact, a beamsplitter gate is a passive (i.e., Fock-number-preserving) operation, and in optical quantum computing, it can be implemented simply by routing photons through physical optical elements such as glass plates and fiber-optic couplers. On the other hand, the CZ gate is a non-passive operation that typically requires active nonlinear optical media, squeezers, or measurement-based feedforward protocols to generate inter-mode interactions, resulting in a more complex decomposition into the standard set of basic gates.

---

## 1. Setup

We import the required libraries and define the circuit building functions to compare the two generation techniques side by side.
The `build_epr_bs` function is adopted from the first notebook.

In [ ]:
from math import pi

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from claveles.circuit import CircuitRepr
from claveles.circuit.ops.std import (
    BeamSplitter,
    ControlledZ,
    MeasureHomodyne,
    PhaseRotation,
)
from claveles.circuit.state import QuantumState
from claveles.circuit.visualize import make_figure
from claveles.client import SimulatorClient
from claveles.execute import execute

In [ ]:
r = 3.0  # squeezing parameter
n_shots = 2000
hbar = 1.0

sns.set_theme(style="whitegrid")

In [ ]:
def build_epr_bs(r: float, name: str = "EPR via BeamSplitter") -> CircuitRepr:
    c = CircuitRepr(name)
    c.Q(0) | QuantumState.squeezed(r=r, phi=0)  # x-squeezed
    c.Q(1) | QuantumState.squeezed(r=r, phi=pi / 2)  # p-squeezed
    c.Q(0, 1) | BeamSplitter(theta=pi / 4, phi=pi)
    return c

---

## 2. Generating the EPR State with a CZ Gate

Two $p$-squeezed vacuum modes are coupled via a CV CZ gate with a gain parameter $g = 1$, followed by a phase rotation of $-\pi/2$ applied to mode 1.

In [ ]:
def build_epr_cz(r: float, name: str = "EPR via CZ gate") -> CircuitRepr:
    c = CircuitRepr(name)
    c.Q(0) | QuantumState.squeezed(r=r, phi=pi / 2)  # p-squeezed
    c.Q(1) | QuantumState.squeezed(r=r, phi=pi / 2)  # p-squeezed
    c.Q(0, 1) | ControlledZ(g=1)
    c.Q(1) | PhaseRotation(phi=-pi / 2)
    return c

In [ ]:
circuit_cz = build_epr_cz(r=r)
fig = make_figure(circuit_cz)
fig.show()

For reference, the beamsplitter-based circuit configuration is shown below.

In [ ]:
circuit_bs = build_epr_bs(r=r)
fig = make_figure(circuit_bs)
fig.show()

---

## 3. Distribution of the Difference and Sum Quadratures

EPR correlations manifest most directly in the difference quadrature $x_0 - x_1$ and the sum quadrature $p_0 + p_1$, both of which should tightly center around zero.
Let us collect homodyne measurement samples from the state generated via the CZ gate.

In [ ]:
def measure_xp(circuit_fn, r: float, n_shots: int = 2000):
    c_x = circuit_fn(r=r)
    c_x.Q(0) | MeasureHomodyne(phi=0)
    c_x.Q(1) | MeasureHomodyne(phi=0)

    c_p = circuit_fn(r=r)
    c_p.Q(0) | MeasureHomodyne(phi=pi / 2)
    c_p.Q(1) | MeasureHomodyne(phi=pi / 2)

    client = SimulatorClient(n_shots=n_shots, backend="cpu", state_save_policy="none")

    res_x = execute(c_x, client)
    res_p = execute(c_p, client)

    x0 = np.array([res_x.client_result[i].get_value(0) for i in range(n_shots)])
    x1 = np.array([res_x.client_result[i].get_value(1) for i in range(n_shots)])
    p0 = np.array([res_p.client_result[i].get_value(0) for i in range(n_shots)])
    p1 = np.array([res_p.client_result[i].get_value(1) for i in range(n_shots)])
    return x0, x1, p0, p1

In [ ]:
x0, x1, p0, p1 = measure_xp(build_epr_cz, r=r, n_shots=n_shots)
diff_x = x0 - x1
sum_p = p0 + p1

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(diff_x, bins=50, density=True, color="steelblue", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="ideal: 0")
axes[0].set_xlabel(r"$x_0 - x_1$")
axes[0].set_ylabel("probability density")
axes[0].set_title(r"Distribution of $x_0 - x_1$")
axes[0].legend()

axes[1].hist(sum_p, bins=50, density=True, color="darkorange", alpha=0.8)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5, label="ideal: 0")
axes[1].set_xlabel(r"$p_0 + p_1$")
axes[1].set_ylabel("probability density")
axes[1].set_title(r"Distribution of $p_0 + p_1$")
axes[1].legend()

fig.suptitle(f"EPR difference and sum quadratures (CZ method, $r = {r}$)")
fig.tight_layout()
plt.show()

In [ ]:
print(f"Var(x0 - x1) = {diff_x.var():.4f}  (theory: {np.exp(-2 * r) * hbar / 2:.4f})")
print(f"Var(p0 + p1) = {sum_p.var():.4f}  (theory: {np.exp(-2 * r) * hbar / 2:.4f})")

For the CZ gate method, the theoretical variance of both $x_0 - x_1$ and $p_0 + p_1$ is given by $e^{-2r}\hbar/2$.
As the squeezing parameter $r$ increases, both distributions narrow significantly around zero, converging toward ideal EPR correlations.

---

## 4. Comparing the Two Generation Methods

Both methods successfully establish the same EPR correlation structure ($\hat{x}_0 \approx \hat{x}_1$ and $\hat{p}_0 \approx -\hat{p}_1$); however, their variances differ by a factor of two.
The beamsplitter method yields a variance of $e^{-2r}\hbar$, whereas the CZ method produces $e^{-2r}\hbar/2$.
We validate these results against their respective theoretical predictions.

In [ ]:
def measure_epr_correlations(circuit_fn, r: float, n_shots: int = 2000):
    x0, x1, p0, p1 = measure_xp(circuit_fn, r=r, n_shots=n_shots)
    return (x0 - x1).var(), (p0 + p1).var()

In [ ]:
var_x_bs, var_p_bs = measure_epr_correlations(build_epr_bs, r=r)
var_x_cz, var_p_cz = measure_epr_correlations(build_epr_cz, r=r)
theory_bs = np.exp(-2 * r) * hbar
theory_cz = np.exp(-2 * r) * hbar / 2

print(f"{'Method':<10}  Var(x0-x1)   Var(p0+p1)   Theory")
print(f"{'BS':<10}  {var_x_bs:.4f}       {var_p_bs:.4f}       {theory_bs:.4f}")
print(f"{'CZ':<10}  {var_x_cz:.4f}       {var_p_cz:.4f}       {theory_cz:.4f}")

The numerical results confirm that both methods reproduce the expected EPR correlation structure.
Crucially, the variance from the CZ method is exactly half that of the beamsplitter approach, perfectly matching the theoretical predictions of $e^{-2r}\hbar/2$ versus $e^{-2r}\hbar$.

---

## 5. Squeezing Strength and Correlation Quality

Increasing the squeezing strength directly enhances the quality of the EPR correlations.
We perform a parameter sweep over $r$ and record the variance of the difference quadrature $x_0 - x_1$ for both implementation techniques.

In [ ]:
r_vals = np.linspace(0.5, 4.0, 8)
vars_bs = []
vars_cz = []

for r_val in r_vals:
    vx_bs, _ = measure_epr_correlations(build_epr_bs, r=r_val, n_shots=1000)
    vx_cz, _ = measure_epr_correlations(build_epr_cz, r=r_val, n_shots=1000)
    vars_bs.append(vx_bs)
    vars_cz.append(vx_cz)
    print(
        f"r={r_val:.2f}  BS: {vx_bs:.4f} (theory {np.exp(-2 * r_val) * hbar:.4f})  "
        f"CZ: {vx_cz:.4f} (theory {np.exp(-2 * r_val) * hbar / 2:.4f})"
    )

In [ ]:
theory_bs = np.exp(-2 * r_vals) * hbar
theory_cz = np.exp(-2 * r_vals) * hbar / 2

plt.figure(figsize=(7, 4))
plt.plot(r_vals, vars_bs, "o-", color="C0", label="BS method")
plt.plot(r_vals, vars_cz, "s-", color="C1", label="CZ method")
plt.plot(
    r_vals, theory_bs, "--", color="C0", alpha=0.6, label=r"BS theory $e^{-2r}\hbar$"
)
plt.plot(
    r_vals, theory_cz, "--", color="C1", alpha=0.6, label=r"CZ theory $e^{-2r}\hbar/2$"
)
plt.xlabel("squeezing parameter $r$")
plt.ylabel(r"$\mathrm{Var}(x_0 - x_1)$")
plt.title("EPR correlation quality vs. squeezing")
plt.legend()
plt.tight_layout()
plt.show()

As $r$ increases, the variance decays exponentially on the scale of $e^{-2r}$, demonstrating a clear convergence toward ideal EPR correlations.
Both experimental datasets tightly follow their respective theoretical curves, with the CZ gate data remaining consistently a factor of two lower than the beamsplitter data.

---

## Summary

This tutorial demonstrated the following:

1. **Two equivalent generation methods**: A two-mode EPR state can be successfully generated either via a beamsplitter (by mixing an $x$-squeezed vacuum and a $p$-squeezed vacuum on a 50:50 beamsplitter with phase $\phi = \pi$) or via a CZ gate (by coupling two $p$-squeezed vacuum modes followed by a $-\pi/2$ phase rotation on mode 1).
2. **Same correlations, different variance**: While both methods achieve the identical target correlation relations ($\hat{x}_0 \approx \hat{x}_1$ and $\hat{p}_0 \approx -\hat{p}_1$), the variance under the CZ gate method ($e^{-2r}\hbar/2$) is precisely half that of the beamsplitter method ($e^{-2r}\hbar$).
3. **Squeezing controls quality**: The variance of the difference quadrature $x_0 - x_1$ decreases exponentially as a function of the squeezing parameter $r$, with both methods exhibiting strict agreement with their theoretical models.

This foundational framework scales naturally to three-mode, four-mode, and larger continuous-variable cluster states, which serve as the essential building blocks for measurement-based CV quantum computing.

---

## References

- \[1\] A. Einstein, B. Podolsky, and N. Rosen, Can Quantum-Mechanical Description of Physical Reality Be Considered Complete?, [*Phys. Rev.* **47**, 777 (1935)](https://doi.org/10.1103/PhysRev.47.777).
- \[2\] J. S. Bell, On the einstein podolsky rosen paradox, [*Phys. Phys. Fiz* **1**, 195 (1964)](https://doi.org/10.1103/PhysicsPhysiqueFizika.1.195).
- \[3\] M. Gu, C. Weedbrook, N. C. Menicucci, T. C. Ralph, and P. van Loock, Quantum Computing with Continuous-Variable Clusters, [*Phys. Rev. A* **79**, 062318 (2009)](https://doi.org/10.1103/PhysRevA.79.062318).